每当有事件发给 filtered_edit，是谁发事件给filtered_edit的

是 QApplication 发的

回顾事件传播的最前端：

```text
操作系统 捕获键盘按下
    ↓
QApplication 事件循环拿到事件
    ↓
QApplication::notify(receiver, event)
    receiver = 当前焦点控件（focused widget）
    ↓
检查 receiver 有没有安装过滤器 → 调用 eventFilter
    ↓ (放行)
调用 receiver.event(event)
```

`QApplication` 的 `notify()` 方法就是 Qt 事件的**总调度员**：

```python
# Qt 内部简化伪代码
def notify(self, receiver, event):
    # 1. 先检查 receiver 有没有安装过滤器
    for filter_obj in receiver.filters():
        if filter_obj.eventFilter(receiver, event):
            return True   # 被拦截了，到此为止

    # 2. 过滤器都放行，交给 receiver 自己处理
    return receiver.event(event)
```

所以在你的例子里，当 `filtered_edit` 获取了焦点、用户按下按键时：

```text
QApplication.notify(receiver=filtered_edit, event=KeyPress)
    → 发现 filtered_edit 装了过滤器（EventHandlerDemo）
    → 调用 EventHandlerDemo.eventFilter(filtered_edit, KeyPress)
        → 非数字 → return True（拦截）
        → 数字   → super().eventFilter() → return False（放行）
    → 放行后 → filtered_edit.event(KeyPress) → keyPressEvent → 字符显示
```

一句话

`QApplication` 是事件的唯一分发源头，它根据"当前谁拥有焦点"决定 `receiver` 是谁，然后调用 `notify(receiver, event)` 开始整条传播链。

为什么要用EventHandlerDemo的eventFilter，用其他对象的不行吗  
完全可以用其他对象，这里用 `self` 只是因为懒

两种写法对比

**当前写法（用 EventHandlerDemo 自己做过滤器）：**

```python
class EventHandlerDemo(QMainWindow):
    def __init__(self):
        ...
        self.filtered_edit.installEventFilter(self)   # 自己过滤自己

    def eventFilter(self, obj, event):                # 过滤逻辑写在自己身上
        ...
```

**用独立对象做过滤器：**

```python
class DigitOnlyFilter(QObject):
    def eventFilter(self, obj, event):
        if event.type() == QEvent.Type.KeyPress:
            key = event.key()
            allowed = {Qt.Key.Key_0, ...}
            if key not in allowed:
                return True
        return super().eventFilter(obj, event)


class EventHandlerDemo(QMainWindow):
    def __init__(self):
        ...
        filter_obj = DigitOnlyFilter()                   # 独立的过滤器对象
        self.filtered_edit.installEventFilter(filter_obj) # 装了另一个对象
        # 不需要再写 eventFilter 方法了
```

### 对比

| | 用 `self` | 用独立对象 |
|---|---|---|
| 代码量 | 少，不需要新建类 | 要多定义一个类 |
| 能访问主窗囗其他控件？ | 容易（`self.status_label` 等） | 需要传引用或信号 |
| 职责清晰度 | 混在主窗囗类里 | 职责单一，独立可复用 |
| 适合场景 | 简单场景，教学演示 | 过滤器逻辑复杂、需要复用 |

### 为什么教学代码选 `self`？

因为简单——不需要多写一个类，初学者一眼看到 `eventFilter` 就在同一个文件里，不用跳来跳去。但工程实践中，独立过滤器对象更清晰，可以单独测试和复用。